# Gold Set Evaluation & Feedback Injection

Compare LLM classification accuracy **before** and **after** injecting a curated SME gold set as few-shot feedback examples.

**Workflow:**
1. Load the SME gold set and inspect label distribution
2. Run baseline classification on gold set records (no feedback)
3. Measure baseline accuracy / precision / recall / F1
4. Import a stratified subset of the gold set into the `FeedbackStore` as few-shot corrections
5. Re-run classification with few-shot examples injected into every prompt
6. Compare metrics — see the accuracy lift from feedback injection
7. Record-level analysis: improvements, regressions, persistent errors
8. Export a comparison report

**Gold set format expected:**
| Column | Description |
|---|---|
| `uid` | Unique record identifier |
| `transcript` | The text to classify |
| `correct_contact_reason` | SME-assigned correct contact reason code |
| `correct_root_cause` | SME-assigned correct root cause code |
| `explanation` | SME rationale for the classification |
| `new_contact_reason_if_other` | *(optional)* More specific label when code is 'other' |
| `new_root_cause_if_other` | *(optional)* More specific label when code is 'other' |

## 0 · Imports

In [ ]:
WORKSPACE = "_template"  # folder under use_cases/ to work in — change to run on another use case
import sys, os
from pathlib import Path

# Ensure project root is on sys.path and is the working directory
_root = Path().absolute()
# Navigate up until we find the rca_pipeline package
while not (_root / "use_cases").is_dir() and _root.parent != _root:
    _root = _root.parent
_root = _root / "use_cases" / WORKSPACE  # the use-case folder this notebook works in
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"Working directory: {os.getcwd()}")


In [ ]:
import asyncio
import json
import time
import warnings
from dataclasses import replace as _replace
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
import logging

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s │ %(levelname)-7s │ %(name)-20s │ %(message)s",
    datefmt="%H:%M:%S",
)
logging.getLogger("pipeline").setLevel(logging.INFO)
logging.getLogger("nb_gold_eval").setLevel(logging.INFO)
logger = logging.getLogger("nb_gold_eval")
warnings.filterwarnings("ignore", category=UserWarning)

from rca_pipeline.config import RunConfig, get_credentials
from rca_pipeline.taxonomy import load_taxonomies
from rca_pipeline.utils import drop_enrichment_cols, post_process, write_output_file
from rca_pipeline.pipeline import prepare_batched_records, run_classification_module, MODULE_REGISTRY
from rca_pipeline.llm_client import (
    set_system_prompt_suffix,
    clear_system_prompt_suffix,
    register_cost_tracker,
    unregister_cost_tracker,
    set_cost_tracker_module,
)
from rca_pipeline.enhancements.feedback import FeedbackStore

try:
    from rca_pipeline.enhancements.confidence import enrich_with_confidence, flag_low_confidence, inject_confidence_instruction
    _HAS_CONFIDENCE = True
except ImportError:
    _HAS_CONFIDENCE = False

try:
    from rca_pipeline.enhancements.cost_tracker import CostTracker
    _HAS_COST = True
except ImportError:
    _HAS_COST = False

print(f"Registered modules: {list(MODULE_REGISTRY.keys())}")
print(f"Confidence scoring: {'✓' if _HAS_CONFIDENCE else '✗'}")
print(f"Cost tracking:      {'✓' if _HAS_COST else '✗'}")

## 1 · Configuration

Edit the variables in this cell before running the notebook.

In [ ]:
# ── Gold set path & column names ──────────────────────────────────────────
GOLD_SET_FILE = "data/gold_set.xlsx"   # supports .xlsx and .csv

UID_COL  = "uid"
TEXT_COL = "transcript"

# Map each module to the column names in YOUR gold set.
#   pipeline_field  : column name in the enriched pipeline output (e.g. contact_reason_primary_code)
#   code_col        : column in your gold set with the correct taxonomy code
#   explanation_col : SME explanation column (used as the correction reason text)
#   other_note_col  : optional — SME's more specific label when the correct code is 'other'
GOLD_COLUMNS = {
    "contact_reason": {
        "pipeline_field":  "contact_reason_primary_code",
        "code_col":        "correct_contact_reason",
        "explanation_col": "explanation",
        "other_note_col":  "new_contact_reason_if_other",  # set to None if absent
    },
    "root_cause": {
        "pipeline_field":  "root_cause_primary_code",
        "code_col":        "correct_root_cause",
        "explanation_col": "explanation",
        "other_note_col":  "new_root_cause_if_other",      # set to None if absent
    },
}

# ── Modules to evaluate ───────────────────────────────────────────────────
# Only modules registered in MODULE_REGISTRY (built by rca build) can run.
# Add or remove module names to match what's in your gold set.
MODULES = [m for m in GOLD_COLUMNS if m in MODULE_REGISTRY]

# ── Taxonomy directory ────────────────────────────────────────────────────
TAXONOMY_DIR = "taxonomies"  # e.g. "taxonomies_scopus_content" for a product-specific run

# ── Gold set split ────────────────────────────────────────────────────────
# FEW_SHOT_RATIO: fraction of gold set used as few-shot examples.
# The rest is held out as an evaluation set not seen by the LLM.
FEW_SHOT_RATIO = 0.7
RANDOM_SEED    = 42

# ── Feedback settings ─────────────────────────────────────────────────────
FEEDBACK_DIR          = "feedback/gold_set/"  # isolated dir so it doesn't mix with production feedback
FEEDBACK_STRATEGY     = "diverse"             # 'diverse' maximises taxonomy coverage for a curated set
FEEDBACK_MAX_EXAMPLES = 15                    # higher than default since gold set is curated

# ── Pipeline settings ─────────────────────────────────────────────────────
BASELINE_OUTPUT   = None   # None = always run fresh; or set to path of an existing XLSX to skip the run
BASELINE_OUT_FILE = "runs/output/gold_eval_baseline.xlsx"
ENHANCED_OUT_FILE = "runs/output/gold_eval_enhanced.xlsx"
COMPARISON_REPORT = "runs/output/gold_eval_comparison.xlsx"

MODEL          = "azure/dsaic-llm-general"  # override if needed
BATCH_SIZE     = 10
CONCURRENCY    = 2

ENABLE_CONFIDENCE    = True
CONFIDENCE_THRESHOLD = 0.70

TRACK_COSTS     = True
COST_REPORT_DIR = "runs/cost_reports"

# ── Display settings ──────────────────────────────────────────────────────
TOP_N_CLASSES = 15  # max classes shown in confusion matrix heatmap

# ── Sanity checks ─────────────────────────────────────────────────────────
creds   = get_credentials()
missing = [k for k, v in creds.items() if not v]
print("Missing credentials:", missing if missing else "None ✓")
print(f"Modules to evaluate: {MODULES}")
if not MODULES:
    print("⚠️  No matching modules found in MODULE_REGISTRY!")
    print(f"   MODULE_REGISTRY keys: {list(MODULE_REGISTRY.keys())}")
    print(f"   GOLD_COLUMNS keys:    {list(GOLD_COLUMNS.keys())}")
    print("   → Run rca build first, or adjust GOLD_COLUMNS keys to match.")

## 2 · Helper functions

In [ ]:
# ── Pipeline runner ───────────────────────────────────────────────────────
async def _run_pipeline(df_in, cfg, modules, taxonomies, feedback_prompts=None):
    """
    Run specified modules on df_in (in-memory DataFrame), return enriched DataFrame.
    Modules run sequentially to avoid the global system-prompt-suffix race condition
    that arises when multiple modules with different prompts run concurrently.
    """
    confidence_instr = ""
    if cfg.enable_confidence and _HAS_CONFIDENCE:
        confidence_instr = inject_confidence_instruction("")

    cost_tracker = None
    if cfg.track_costs and _HAS_COST:
        cost_tracker = CostTracker(model=cfg.model)
        register_cost_tracker(cost_tracker)

    batches = prepare_batched_records(df_in, cfg)
    all_results = []

    for mod in modules:
        suffix_parts = [
            p for p in [
                confidence_instr,
                (feedback_prompts or {}).get(mod, ""),
            ] if p
        ]
        set_system_prompt_suffix("\n".join(suffix_parts))
        set_cost_tracker_module(mod)

        results = await run_classification_module(mod, batches, taxonomies, cfg)
        if cfg.enable_confidence and _HAS_CONFIDENCE:
            results = enrich_with_confidence(results, threshold=cfg.confidence_threshold)
        all_results.extend(results)

    clear_system_prompt_suffix()

    if cost_tracker is not None:
        cost_tracker.print_summary()
        Path(cfg.cost_report_dir).mkdir(parents=True, exist_ok=True)
        cost_tracker.save(Path(cfg.cost_report_dir) / f"costs_{cost_tracker._run_id}.json")
        unregister_cost_tracker()

    # Merge by UID
    grouped = {}
    for r in all_results:
        uid = str(r.get("uid", ""))
        grouped.setdefault(uid, {"uid": uid}).update(r)

    df_enriched = post_process(df_in, list(grouped.values()), taxonomies, cfg)
    if cfg.enable_confidence and _HAS_CONFIDENCE and "_confidence" in df_enriched.columns:
        df_enriched = flag_low_confidence(df_enriched, threshold=cfg.confidence_threshold)
    return df_enriched


# ── Metrics ───────────────────────────────────────────────────────────────
def _compute_metrics(df_enriched, gold_df, pipeline_field, gold_col, uid_col):
    """Merge pipeline output with gold standard and compute classification metrics."""
    left  = df_enriched[[uid_col, pipeline_field]].copy()
    left[uid_col] = left[uid_col].astype(str)
    right = gold_df[[uid_col, gold_col]].copy()
    right[uid_col] = right[uid_col].astype(str)

    merged = left.merge(right, on=uid_col, how="inner")

    y_pred = merged[pipeline_field].fillna("").astype(str).str.strip().str.lower()
    y_true = merged[gold_col].fillna("").astype(str).str.strip().str.lower()

    # Keep only rows where gold has a non-empty answer
    valid  = (y_true != "") & (y_true != "nan")
    y_pred = y_pred[valid].reset_index(drop=True)
    y_true = y_true[valid].reset_index(drop=True)
    merged = merged[valid].reset_index(drop=True)

    n = len(y_true)
    if n == 0:
        return {"accuracy": 0.0, "report": {}, "y_true": y_true, "y_pred": y_pred, "merged": merged, "n": 0}

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "report":   classification_report(y_true, y_pred, output_dict=True, zero_division=0),
        "y_true":   y_true,
        "y_pred":   y_pred,
        "merged":   merged,
        "n":        n,
    }


def _summary_metrics(m):
    """Extract scalar summary from a _compute_metrics result dict."""
    r = m.get("report", {})
    return {
        "n":                  m["n"],
        "accuracy":           m["accuracy"],
        "macro_f1":           r.get("macro avg",    {}).get("f1-score",  0.0),
        "weighted_f1":        r.get("weighted avg", {}).get("f1-score",  0.0),
        "macro_precision":    r.get("macro avg",    {}).get("precision", 0.0),
        "macro_recall":       r.get("macro avg",    {}).get("recall",    0.0),
        "weighted_precision": r.get("weighted avg", {}).get("precision", 0.0),
        "weighted_recall":    r.get("weighted avg", {}).get("recall",    0.0),
    }


# ── Confusion matrix heatmap ──────────────────────────────────────────────
def _confusion_heatmap(y_true, y_pred, title, top_n=TOP_N_CLASSES, ax=None):
    """Plot a confusion matrix for the top-N most frequent true classes."""
    top_classes = y_true.value_counts().head(top_n).index.tolist()
    # Include any predicted class not in top_n as a bin so the matrix stays square
    all_classes = sorted(set(top_classes) | (set(y_pred[y_true.isin(top_classes)]) & set(top_classes)))
    mask = y_true.isin(all_classes)
    yt, yp = y_true[mask], y_pred[mask]

    cm = confusion_matrix(yt, yp, labels=all_classes)
    dim = len(all_classes)
    if ax is None:
        _, ax = plt.subplots(figsize=(max(8, dim), max(6, dim * 0.75)))

    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=all_classes, yticklabels=all_classes,
        ax=ax, linewidths=0.4, cbar=False,
    )
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("True (Gold)", fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.tick_params(axis="y", rotation=0, labelsize=8)
    return ax


print("Helper functions defined ✓")

## 3 · Load & inspect gold set

In [ ]:
_p = Path(GOLD_SET_FILE)
if _p.suffix.lower() in (".xlsx", ".xls"):
    gold_df = pd.read_excel(_p, engine="openpyxl")
elif _p.suffix.lower() == ".csv":
    gold_df = pd.read_csv(_p)
else:
    raise ValueError(f"Unsupported file type: {_p.suffix}  (expected .xlsx, .xls, or .csv)")

gold_df[UID_COL] = gold_df[UID_COL].astype(str)
print(f"Gold set: {len(gold_df):,} records")
print(f"Columns : {list(gold_df.columns)}")

# ── Column validation ─────────────────────────────────────────────────────
errors = []
for col in [UID_COL, TEXT_COL]:
    if col not in gold_df.columns:
        errors.append(f"Required column '{col}' not found")
for mod, mc in GOLD_COLUMNS.items():
    if mod not in MODULES:
        continue
    for fk in ["code_col", "explanation_col"]:
        col = mc.get(fk)
        if col and col not in gold_df.columns:
            errors.append(f"Module '{mod}': column '{col}' (key: {fk}) not found in gold set")

if errors:
    print("\n❌ Column errors:")
    for e in errors:
        print(f"   {e}")
    print("\n→ Fix GOLD_COLUMNS in the Configuration cell and re-run.")
    raise ValueError("Column mapping errors — see above.")
print("\n✓ All required columns present")

# ── Deduplication ─────────────────────────────────────────────────────────
dups = gold_df.duplicated(subset=[UID_COL])
if dups.any():
    print(f"⚠️  {dups.sum()} duplicate UIDs — keeping last occurrence")
    gold_df = gold_df.drop_duplicates(subset=[UID_COL], keep="last").reset_index(drop=True)

# ── Label distribution ────────────────────────────────────────────────────
print()
for mod in MODULES:
    mc = GOLD_COLUMNS[mod]
    gc = mc["code_col"]
    print(f"{'─'*60}")
    print(f"{mod}  |  gold column: {gc}")
    vc = gold_df[gc].value_counts(dropna=False)
    for val, cnt in vc.items():
        pct = cnt / len(gold_df) * 100
        print(f"  {str(val):35s} {cnt:4d}  ({pct:5.1f}%)  {'█' * int(pct / 2)}")

    other_col = mc.get("other_note_col")
    if other_col and other_col in gold_df.columns:
        with_notes = (
            gold_df[other_col].notna()
            & (gold_df[other_col].astype(str).str.strip() != "")
            & (gold_df[other_col].astype(str).str.lower() != "nan")
        )
        print(f"  → Records with 'other' supplementary notes: {with_notes.sum()}")

print(f"{'─'*60}")
print("\nSample (first 3 rows):")
_show_cols = [UID_COL, TEXT_COL] + [mc["code_col"] for mc in GOLD_COLUMNS.values() if mc["code_col"] in gold_df.columns]
display(gold_df[_show_cols].head(3))

## 4 · Baseline classification *(no feedback)*

Classify the gold set records with **no few-shot corrections** injected.
This gives us the LLM's unassisted accuracy, which we'll compare against later.

Set `BASELINE_OUTPUT` in the configuration cell to a path if you already have a baseline run and want to skip this step.

In [ ]:
taxonomies = load_taxonomies(TAXONOMY_DIR)
loaded = [k for k, v in taxonomies.items() if v]
empty  = [k for k, v in taxonomies.items() if not v]
print(f"Loaded taxonomies : {loaded}")
if empty:
    print(f"Empty/missing     : {empty}")
missing_tax = [m for m in MODULES if m not in taxonomies or not taxonomies[m]]
if missing_tax:
    print(f"\n⚠️  Modules without taxonomy: {missing_tax}")
    print("   → Check TAXONOMY_DIR or run rca build first.")

In [ ]:
_base_cfg = RunConfig(
    output_file=BASELINE_OUT_FILE,
    taxonomy_dir=TAXONOMY_DIR,
    id_col=UID_COL,
    text_col=TEXT_COL,
    mode="classify",
    modules=MODULES,
    model=MODEL,
    batch_size=BATCH_SIZE,
    concurrency=CONCURRENCY,
    enable_confidence=ENABLE_CONFIDENCE,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    feedback_dir=None,        # ← no feedback for baseline
    feedback_max_examples=0,
    track_costs=TRACK_COSTS,
    cost_report_dir=COST_REPORT_DIR,
)

Path("output").mkdir(parents=True, exist_ok=True)

if BASELINE_OUTPUT and Path(BASELINE_OUTPUT).exists():
    print(f"Loading existing baseline from: {BASELINE_OUTPUT}")
    df_baseline = pd.read_excel(BASELINE_OUTPUT, engine="openpyxl")
    df_baseline[UID_COL] = df_baseline[UID_COL].astype(str)
    print(f"Loaded {len(df_baseline):,} rows × {len(df_baseline.columns)} columns")
else:
    print(f"Running baseline classification on {len(gold_df):,} gold set records…")
    _t0 = time.time()
    df_baseline = await _run_pipeline(
        drop_enrichment_cols(gold_df[[UID_COL, TEXT_COL]]),
        _base_cfg,
        MODULES,
        taxonomies,
        feedback_prompts=None,
    )
    write_output_file(df_baseline, BASELINE_OUT_FILE)
    print(f"\n✓ Baseline saved → {BASELINE_OUT_FILE}")
    print(f"  {len(df_baseline):,} rows × {len(df_baseline.columns)} columns  ({time.time()-_t0:.1f}s)")

## 5 · Baseline metrics

In [ ]:
baseline_metrics = {}

for mod in MODULES:
    mc = GOLD_COLUMNS[mod]
    pf = mc["pipeline_field"]
    gc = mc["code_col"]

    if pf not in df_baseline.columns:
        print(f"⚠️  '{pf}' not found in baseline output — skipping {mod}")
        print(f"   Available columns: {[c for c in df_baseline.columns if 'code' in c.lower()]}")
        continue

    m  = _compute_metrics(df_baseline, gold_df, pf, gc, UID_COL)
    sm = _summary_metrics(m)
    baseline_metrics[mod] = m

    print(f"\n{'═'*62}")
    print(f"BASELINE — {mod}  (n={sm['n']})")
    print(f"{'─'*62}")
    print(f"  Accuracy           : {sm['accuracy']:.4f}  ({int(sm['accuracy']*sm['n'])}/{sm['n']} correct)")
    print(f"  Macro F1           : {sm['macro_f1']:.4f}")
    print(f"  Weighted F1        : {sm['weighted_f1']:.4f}")
    print(f"  Macro Precision    : {sm['macro_precision']:.4f}")
    print(f"  Macro Recall       : {sm['macro_recall']:.4f}")

    print("\nPer-class report:")
    cr = pd.DataFrame(m["report"]).T
    cr = cr[cr.index.isin(m["y_true"].unique())]
    cr = cr.sort_values("support", ascending=False)
    display(cr[["precision", "recall", "f1-score", "support"]].round(3))

# ── Confusion matrices ────────────────────────────────────────────────────
for mod, m in baseline_metrics.items():
    fig, ax = plt.subplots(figsize=(12, 9))
    _confusion_heatmap(m["y_true"], m["y_pred"], f"Baseline — {mod}  (top {TOP_N_CLASSES} classes)", ax=ax)
    plt.tight_layout()
    plt.show()

In [ ]:
# Top misclassified pairs — useful for understanding systematic errors
for mod, m in baseline_metrics.items():
    errors_mask = m["y_true"] != m["y_pred"]
    n_errors = errors_mask.sum()
    print(f"\n{'─'*62}")
    print(f"Baseline error analysis — {mod}  ({n_errors}/{m['n']} wrong)")
    if n_errors == 0:
        print("  Perfect baseline — no errors.")
        continue

    err_df = pd.DataFrame({"true": m["y_true"][errors_mask].values, "pred": m["y_pred"][errors_mask].values})
    pairs  = err_df.groupby(["true", "pred"]).size().sort_values(ascending=False).head(12)
    print(f"  Top misclassified pairs (true → predicted):")
    for (true, pred), cnt in pairs.items():
        pct = cnt / n_errors * 100
        print(f"  {true:35s} → {pred:35s}  {cnt:3d}x  ({pct:4.1f}% of errors)")

## 6 · Prepare gold set feedback

Split the gold set into a **few-shot set** (used to teach the LLM) and a **held-out eval set** (never seen by the LLM, used for unbiased evaluation).

Corrections are built by comparing each gold set record's correct code against the baseline prediction:
- **Wrong → right** corrections show the LLM how it previously erred and what the right answer is
- **'Other' anchoring examples** are included even when baseline was correct, because the SME's supplementary note teaches the LLM the expected level of specificity

In [ ]:
# Stratified split on the first module's gold code column
_strat_col   = GOLD_COLUMNS[MODULES[0]]["code_col"]
_strat_vals  = gold_df[_strat_col].astype(str)
_code_counts = _strat_vals.value_counts()

# Classes with only 1 example can't be stratified — put them in few-shot set
_singletons   = gold_df[_strat_vals.isin(_code_counts[_code_counts < 2].index)]
_stratifiable = gold_df[~_strat_vals.isin(_code_counts[_code_counts < 2].index)]

if len(_stratifiable) >= 2:
    try:
        few_shot_df, eval_df = train_test_split(
            _stratifiable,
            test_size=round(1 - FEW_SHOT_RATIO, 2),
            random_state=RANDOM_SEED,
            stratify=_stratifiable[_strat_col],
        )
        # Singletons → few-shot (better to show as an example than to evaluate a one-off class)
        few_shot_df = pd.concat([few_shot_df, _singletons], ignore_index=True)
    except ValueError as e:
        print(f"Stratified split failed ({e}) — falling back to random split")
        few_shot_df, eval_df = train_test_split(_stratifiable, test_size=round(1-FEW_SHOT_RATIO, 2), random_state=RANDOM_SEED)
        few_shot_df = pd.concat([few_shot_df, _singletons], ignore_index=True)
else:
    print("⚠️  Gold set too small to split — using full set for both few-shot and evaluation")
    few_shot_df = gold_df.copy()
    eval_df     = gold_df.copy()

print(f"Gold set split (stratified on '{_strat_col}'):")
print(f"  Few-shot examples : {len(few_shot_df):,}  ({len(few_shot_df)/len(gold_df)*100:.0f}%)")
print(f"  Held-out eval     : {len(eval_df):,}  ({len(eval_df)/len(gold_df)*100:.0f}%)")
print(f"  Singletons → few-shot: {len(_singletons)}")

In [ ]:
# Build a lookup of baseline predictions for the few-shot records
baseline_lookup = {
    str(row[UID_COL]): row
    for _, row in df_baseline.iterrows()
}

# Clear any previous gold-set corrections so each notebook run is idempotent
Path(FEEDBACK_DIR).mkdir(parents=True, exist_ok=True)
for mod in MODULES:
    fp = Path(FEEDBACK_DIR) / f"{mod}_corrections.jsonl"
    if fp.exists():
        fp.unlink()
        print(f"Cleared: {fp}")

store = FeedbackStore(FEEDBACK_DIR)
import_summary = {}

for mod in MODULES:
    mc      = GOLD_COLUMNS[mod]
    pf      = mc["pipeline_field"]
    gc      = mc["code_col"]
    exp_col = mc.get("explanation_col")
    other_col = mc.get("other_note_col")

    corrections = []
    n_wrong_to_right = 0
    n_other_anchoring = 0

    for _, row in few_shot_df.iterrows():
        uid       = str(row[UID_COL])
        text      = str(row.get(TEXT_COL, ""))
        gold_code = str(row.get(gc, "")).strip()

        if not gold_code or gold_code.lower() in ("nan", ""):
            continue

        baseline_row  = baseline_lookup.get(uid, {})
        baseline_code = (
            str(baseline_row.get(pf, "")).strip()
            if hasattr(baseline_row, "get") else ""
        )

        # Build reason text from SME explanation
        reason = ""
        if exp_col and exp_col in row.index:
            raw = str(row.get(exp_col, "")).strip()
            if raw.lower() not in ("nan", ""):
                reason = raw

        # Append 'other' supplementary note — teaches LLM what granularity is expected
        has_other_note = False
        if other_col and other_col in row.index:
            note = str(row.get(other_col, "")).strip()
            if note and note.lower() not in ("nan", ""):
                reason = (reason + ". " if reason else "") + f"More specific label: {note}"
                reason = reason.strip(". ")
                has_other_note = True

        baseline_correct = baseline_code.lower() == gold_code.lower()
        if not baseline_correct:
            n_wrong_to_right += 1

        # Include record as a correction if:
        #   (a) baseline was wrong — shows the LLM what NOT to do and what IS correct
        #   (b) baseline was 'other' with a note — anchors edge-case behaviour regardless
        if not baseline_correct or has_other_note:
            if has_other_note and baseline_correct:
                n_other_anchoring += 1
            corrections.append({
                "uid":       uid,
                "text":      text,
                "original":  {pf: baseline_code or "(not classified)"},
                "corrected": {pf: gold_code},
                "reason":    reason,
            })

    count = store.add_corrections_batch(mod, corrections)
    import_summary[mod] = {
        "total": count,
        "wrong_to_right": n_wrong_to_right,
        "other_anchoring": n_other_anchoring,
    }
    print(f"\n{mod}: {count} corrections imported")
    print(f"  Wrong → right corrections : {n_wrong_to_right}")
    print(f"  'Other' anchoring examples: {n_other_anchoring}")

# Preview what will be injected
print()
for mod in MODULES:
    print(f"{'─'*62}")
    print(f"Few-shot block preview — {mod} (strategy: {FEEDBACK_STRATEGY}):")
    preview = store.build_few_shot_prompt(
        mod, max_examples=3, max_text_chars=150, selection_strategy=FEEDBACK_STRATEGY
    )
    if preview:
        for line in preview.strip().split("\n")[:20]:
            print(f"  {line[:110]}")
        remaining = len(preview.strip().split("\n")) - 20
        if remaining > 0:
            print(f"  … ({remaining} more lines)")
    else:
        print("  (no corrections stored — all baseline predictions matched gold)")

## 7 · Enhanced classification *(with feedback)*

Re-run the same records with the gold set corrections injected as few-shot examples.

In [ ]:
_enh_cfg = _replace(
    _base_cfg,
    output_file=ENHANCED_OUT_FILE,
    feedback_dir=FEEDBACK_DIR,
    feedback_max_examples=FEEDBACK_MAX_EXAMPLES,
    feedback_strategy=FEEDBACK_STRATEGY,
)

# Build per-module few-shot prompt blocks
_feedback_prompts = {}
for mod in MODULES:
    fp = store.build_few_shot_prompt(
        mod, max_examples=FEEDBACK_MAX_EXAMPLES, selection_strategy=FEEDBACK_STRATEGY
    )
    if fp:
        _feedback_prompts[mod] = fp
        print(f"{mod}: {len(fp)}-char few-shot block ({FEEDBACK_MAX_EXAMPLES} max examples, strategy='{FEEDBACK_STRATEGY}')")
    else:
        print(f"{mod}: no few-shot examples (baseline was already perfect on few-shot set)")

print(f"\nRunning enhanced classification on {len(gold_df):,} records…")
_t0 = time.time()
df_enhanced = await _run_pipeline(
    drop_enrichment_cols(gold_df[[UID_COL, TEXT_COL]]),
    _enh_cfg,
    MODULES,
    taxonomies,
    feedback_prompts=_feedback_prompts,
)
write_output_file(df_enhanced, ENHANCED_OUT_FILE)
print(f"\n✓ Enhanced output saved → {ENHANCED_OUT_FILE}")
print(f"  {len(df_enhanced):,} rows × {len(df_enhanced.columns)} columns  ({time.time()-_t0:.1f}s)")

In [ ]:
# Inspect the exact few-shot text injected — verify it looks correct before interpreting results
for mod in MODULES:
    block = _feedback_prompts.get(mod, "")
    print(f"{'═'*70}")
    print(f"INJECTED FEW-SHOT BLOCK — {mod}")
    print(f"{'═'*70}")
    if block:
        print(block[:4000])
        if len(block) > 4000:
            print(f"\n… [{len(block) - 4000} more chars]")
    else:
        print("(none — no corrections were stored for this module)")
    print()

## 8 · Metrics comparison — baseline vs enhanced

In [ ]:
enhanced_metrics = {}

for mod in MODULES:
    mc = GOLD_COLUMNS[mod]
    pf = mc["pipeline_field"]
    gc = mc["code_col"]

    if pf not in df_enhanced.columns:
        print(f"⚠️  '{pf}' not found in enhanced output — skipping {mod}")
        continue

    enhanced_metrics[mod] = _compute_metrics(df_enhanced, gold_df, pf, gc, UID_COL)

# ── Side-by-side summary table ────────────────────────────────────────────
comparison_rows = []
for mod in MODULES:
    if mod not in baseline_metrics or mod not in enhanced_metrics:
        continue
    bm = _summary_metrics(baseline_metrics[mod])
    em = _summary_metrics(enhanced_metrics[mod])

    for metric_name, b_val, e_val in [
        ("accuracy",           bm["accuracy"],           em["accuracy"]),
        ("macro_f1",           bm["macro_f1"],           em["macro_f1"]),
        ("weighted_f1",        bm["weighted_f1"],        em["weighted_f1"]),
        ("macro_precision",    bm["macro_precision"],    em["macro_precision"]),
        ("macro_recall",       bm["macro_recall"],       em["macro_recall"]),
        ("weighted_precision", bm["weighted_precision"], em["weighted_precision"]),
        ("weighted_recall",    bm["weighted_recall"],    em["weighted_recall"]),
    ]:
        delta = e_val - b_val
        comparison_rows.append({
            "module":    mod,
            "metric":    metric_name,
            "baseline":  round(b_val, 4),
            "enhanced":  round(e_val, 4),
            "delta":     round(delta, 4),
            "direction": "↑" if delta > 0.005 else ("↓" if delta < -0.005 else "→"),
            "pct_change": f"{delta/b_val*100:+.1f}%" if b_val > 0 else "n/a",
        })

comp_df = pd.DataFrame(comparison_rows)

def _color_delta(val):
    if isinstance(val, float):
        if val > 0.005:  return "color: green; font-weight: bold"
        if val < -0.005: return "color: red; font-weight: bold"
    return ""

print("Comparison — baseline vs enhanced:")
display(
    comp_df.style
    .applymap(_color_delta, subset=["delta"])
    .format({"baseline": "{:.4f}", "enhanced": "{:.4f}", "delta": "{:+.4f}"})
)

In [ ]:
# ── Bar chart: key metrics before vs after ────────────────────────────────
for mod in MODULES:
    if mod not in baseline_metrics or mod not in enhanced_metrics:
        continue
    bm = _summary_metrics(baseline_metrics[mod])
    em = _summary_metrics(enhanced_metrics[mod])

    metric_keys  = ["accuracy", "macro_f1", "weighted_f1", "macro_recall"]
    metric_labels = ["Accuracy", "Macro F1", "Weighted F1", "Macro Recall"]
    b_vals = [bm[k] for k in metric_keys]
    e_vals = [em[k] for k in metric_keys]

    x     = np.arange(len(metric_labels))
    width = 0.35
    fig, ax = plt.subplots(figsize=(10, 5))

    bars_b = ax.bar(x - width/2, b_vals, width, label="Baseline", color="#4C72B0", alpha=0.85)
    bars_e = ax.bar(x + width/2, e_vals, width, label="Enhanced", color="#55A868", alpha=0.85)

    for bar in bars_b:
        ax.annotate(
            f"{bar.get_height():.3f}",
            xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 3), textcoords="offset points",
            ha="center", va="bottom", fontsize=9, color="#2c4a8a",
        )
    for bar in bars_e:
        ax.annotate(
            f"{bar.get_height():.3f}",
            xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 3), textcoords="offset points",
            ha="center", va="bottom", fontsize=9, color="#2d6a41",
        )

    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels, fontsize=11)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score")
    ax.set_title(f"Baseline vs Enhanced — {mod}", fontsize=13)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

# ── Per-class F1 delta ────────────────────────────────────────────────────
for mod in MODULES:
    if mod not in baseline_metrics or mod not in enhanced_metrics:
        continue
    br = baseline_metrics[mod]["report"]
    er = enhanced_metrics[mod]["report"]

    classes     = [k for k in br if k not in ("accuracy", "macro avg", "weighted avg")]
    class_delta = {
        c: er.get(c, {}).get("f1-score", 0.0) - br.get(c, {}).get("f1-score", 0.0)
        for c in classes
    }
    class_delta = dict(sorted(class_delta.items(), key=lambda x: x[1]))

    fig, ax = plt.subplots(figsize=(10, max(4, len(classes) * 0.45)))
    colors = ["#55A868" if v >= 0 else "#C44E52" for v in class_delta.values()]
    ax.barh(list(class_delta.keys()), list(class_delta.values()), color=colors, alpha=0.85)
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xlabel("F1 Delta  (Enhanced − Baseline)")
    ax.set_title(f"Per-class F1 change — {mod}  (green = improved, red = regressed)", fontsize=12)
    ax.tick_params(axis="y", labelsize=9)
    plt.tight_layout()
    plt.show()

# ── Confusion matrices side-by-side ──────────────────────────────────────
for mod in MODULES:
    if mod not in baseline_metrics or mod not in enhanced_metrics:
        continue
    bm = baseline_metrics[mod]
    em = enhanced_metrics[mod]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 9))
    _confusion_heatmap(bm["y_true"], bm["y_pred"], f"Baseline — {mod}", ax=ax1)
    _confusion_heatmap(em["y_true"], em["y_pred"], f"Enhanced — {mod}", ax=ax2)
    plt.suptitle(f"Confusion matrices — {mod}", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

## 9 · Record-level analysis

Categories: **improvements** (wrong → right), **regressions** (right → wrong ⚠️), **persistent errors** (both wrong).

In [ ]:
record_analysis = {}

for mod in MODULES:
    mc = GOLD_COLUMNS[mod]
    pf = mc["pipeline_field"]
    gc = mc["code_col"]

    if mod not in baseline_metrics or mod not in enhanced_metrics:
        continue

    bm = baseline_metrics[mod]
    em = enhanced_metrics[mod]

    # Align baseline and enhanced predictions on the same UIDs
    b_df = bm["merged"][[UID_COL, pf, gc]].rename(
        columns={pf: "baseline_pred", gc: "gold_code"}
    ).copy()
    b_df[UID_COL] = b_df[UID_COL].astype(str)

    e_df = em["merged"][[UID_COL, pf]].rename(columns={pf: "enhanced_pred"}).copy()
    e_df[UID_COL] = e_df[UID_COL].astype(str)

    joined = b_df.merge(e_df, on=UID_COL, how="inner")
    _norm  = lambda s: s.astype(str).str.strip().str.lower()
    joined["gold_norm"]     = _norm(joined["gold_code"])
    joined["baseline_norm"] = _norm(joined["baseline_pred"])
    joined["enhanced_norm"] = _norm(joined["enhanced_pred"])

    joined["baseline_correct"] = joined["baseline_norm"] == joined["gold_norm"]
    joined["enhanced_correct"] = joined["enhanced_norm"] == joined["gold_norm"]

    improvements  = joined[~joined["baseline_correct"] &  joined["enhanced_correct"]]
    regressions   = joined[ joined["baseline_correct"] & ~joined["enhanced_correct"]]
    persistent    = joined[~joined["baseline_correct"] & ~joined["enhanced_correct"]]
    unchanged_ok  = joined[ joined["baseline_correct"] &  joined["enhanced_correct"]]

    record_analysis[mod] = {
        "improvements":    improvements,
        "regressions":     regressions,
        "persistent_errors": persistent,
        "unchanged_correct":  unchanged_ok,
    }

    n = len(joined)
    print(f"\n{'═'*62}")
    print(f"RECORD-LEVEL — {mod}  (n={n})")
    print(f"{'─'*62}")
    print(f"  ✓ Correct in both  : {len(unchanged_ok):4d}  ({len(unchanged_ok)/n*100:5.1f}%)")
    print(f"  ↑ Improvements     : {len(improvements):4d}  ({len(improvements)/n*100:5.1f}%)  wrong → right")
    print(f"  ↓ Regressions      : {len(regressions):4d}  ({len(regressions)/n*100:5.1f}%)  right → wrong ⚠️")
    print(f"  ✗ Persistent errors: {len(persistent):4d}  ({len(persistent)/n*100:5.1f}%)")
    assert len(improvements)+len(regressions)+len(persistent)+len(unchanged_ok) == n

    _show = [UID_COL, "gold_code", "baseline_pred", "enhanced_pred"]

    if len(improvements) > 0:
        print(f"\n  Improvements (baseline wrong → enhanced correct):")
        display(improvements[[c for c in _show if c in improvements.columns]].head(10))

    if len(regressions) > 0:
        print(f"\n  ⚠️  Regressions (baseline was right, enhanced now wrong):")
        display(regressions[[c for c in _show if c in regressions.columns]].head(10))

    if len(persistent) > 0:
        print(f"\n  Persistent errors (still wrong — prediction changed: {(persistent['baseline_norm'] != persistent['enhanced_norm']).sum()}):")
        display(persistent[[c for c in _show if c in persistent.columns]].head(10))

In [ ]:
# ── 'Other' category deep-dive ────────────────────────────────────────────
for mod in MODULES:
    mc        = GOLD_COLUMNS[mod]
    other_col = mc.get("other_note_col")

    if not other_col or other_col not in gold_df.columns:
        continue

    other_records = gold_df[
        gold_df[other_col].notna()
        & (gold_df[other_col].astype(str).str.strip() != "")
        & (gold_df[other_col].astype(str).str.lower() != "nan")
    ]

    if len(other_records) == 0:
        continue

    print(f"\n{'─'*62}")
    print(f"'Other' category deep-dive — {mod}  ({len(other_records)} records with notes)")

    ra         = record_analysis.get(mod, {})
    other_uids = set(other_records[UID_COL].astype(str))

    for label, key in [
        ("Correct in both",   "unchanged_correct"),
        ("Improvements",      "improvements"),
        ("Regressions",       "regressions"),
        ("Persistent errors", "persistent_errors"),
    ]:
        df_cat = ra.get(key, pd.DataFrame())
        if df_cat.empty:
            n_other = 0
        else:
            n_other = df_cat[UID_COL].astype(str).isin(other_uids).sum()
        print(f"  {label:22s}: {n_other}")

    print("\n  'Other' records detail:")
    display(
        other_records[[UID_COL, mc["code_col"], other_col]]
        .rename(columns={mc["code_col"]: "gold_code", other_col: "sme_note"})
        .head(10)
    )

## 10 · Confidence score comparison

In [ ]:
if not (ENABLE_CONFIDENCE and _HAS_CONFIDENCE):
    print("Confidence scoring disabled. Set ENABLE_CONFIDENCE = True to enable.")
elif "_confidence" not in df_baseline.columns or "_confidence" not in df_enhanced.columns:
    print("⚠️  '_confidence' column not found in output — confidence scoring may have failed.")
else:
    for mod in MODULES:
        print(f"\n{'─'*62}")
        print(f"Confidence — {mod}")

        b_conf = pd.to_numeric(df_baseline["_confidence"], errors="coerce").dropna()
        e_conf = pd.to_numeric(df_enhanced["_confidence"], errors="coerce").dropna()

        print(f"  Baseline: mean={b_conf.mean():.3f}  median={b_conf.median():.3f}  std={b_conf.std():.3f}")
        print(f"  Enhanced: mean={e_conf.mean():.3f}  median={e_conf.median():.3f}  std={e_conf.std():.3f}")

        b_flagged = (b_conf < CONFIDENCE_THRESHOLD).sum()
        e_flagged = (e_conf < CONFIDENCE_THRESHOLD).sum()
        print(f"  Baseline flagged for review: {b_flagged}  ({b_flagged/len(b_conf)*100:.1f}%)")
        print(f"  Enhanced flagged for review: {e_flagged}  ({e_flagged/len(e_conf)*100:.1f}%)")

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))

        # Histogram overlay
        axes[0].hist(b_conf, bins=20, alpha=0.6, label="Baseline", color="#4C72B0")
        axes[0].hist(e_conf, bins=20, alpha=0.6, label="Enhanced", color="#55A868")
        axes[0].axvline(
            CONFIDENCE_THRESHOLD, color="red", linestyle="--",
            linewidth=1.2, label=f"Threshold ({CONFIDENCE_THRESHOLD})",
        )
        axes[0].set_xlabel("Confidence")
        axes[0].set_ylabel("Count")
        axes[0].set_title(f"Confidence distribution — {mod}")
        axes[0].legend()

        # Confidence by outcome (improved / regressed / unchanged)
        ra = record_analysis.get(mod, {})
        imp_uids = set(ra.get("improvements",    pd.DataFrame()).get(UID_COL, pd.Series()).astype(str))
        reg_uids = set(ra.get("regressions",     pd.DataFrame()).get(UID_COL, pd.Series()).astype(str))

        e_conf_df = df_enhanced[[UID_COL, "_confidence"]].copy()
        e_conf_df[UID_COL] = e_conf_df[UID_COL].astype(str)
        e_conf_df["category"] = "unchanged"
        e_conf_df.loc[e_conf_df[UID_COL].isin(imp_uids), "category"] = "improved"
        e_conf_df.loc[e_conf_df[UID_COL].isin(reg_uids), "category"] = "regressed"

        for cat, color in [("improved", "#55A868"), ("unchanged", "#4C72B0"), ("regressed", "#C44E52")]:
            subset = pd.to_numeric(
                e_conf_df[e_conf_df["category"] == cat]["_confidence"], errors="coerce"
            ).dropna()
            if len(subset):
                axes[1].hist(subset, bins=15, alpha=0.6, label=f"{cat} ({len(subset)})", color=color)

        axes[1].axvline(CONFIDENCE_THRESHOLD, color="red", linestyle="--", linewidth=1.2)
        axes[1].set_xlabel("Enhanced Confidence")
        axes[1].set_ylabel("Count")
        axes[1].set_title(f"Enhanced confidence by outcome — {mod}")
        axes[1].legend()

        plt.tight_layout()
        plt.show()
        break  # confidence is shared across modules; plot once

## 11 · Export comparison report

In [ ]:
# ── Build export DataFrames ───────────────────────────────────────────────
summary_rows = []
for mod in MODULES:
    if mod not in baseline_metrics or mod not in enhanced_metrics:
        continue
    bm = _summary_metrics(baseline_metrics[mod])
    em = _summary_metrics(enhanced_metrics[mod])
    for metric in [
        "accuracy", "macro_f1", "weighted_f1",
        "macro_precision", "macro_recall",
        "weighted_precision", "weighted_recall",
    ]:
        b_v, e_v = bm[metric], em[metric]
        delta    = e_v - b_v
        summary_rows.append({
            "module":     mod,
            "metric":     metric,
            "n":          bm["n"],
            "baseline":   round(b_v, 4),
            "enhanced":   round(e_v, 4),
            "delta":      round(delta, 4),
            "pct_change": f"{delta/b_v*100:+.1f}%" if b_v > 0 else "n/a",
            "direction":  "improved" if delta > 0.005 else ("regressed" if delta < -0.005 else "unchanged"),
        })

summary_export = pd.DataFrame(summary_rows)

# ── Write Excel with multiple sheets ─────────────────────────────────────
Path(COMPARISON_REPORT).parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(COMPARISON_REPORT, engine="openpyxl") as writer:
    summary_export.to_excel(writer, sheet_name="summary", index=False)

    for mod in MODULES:
        ra = record_analysis.get(mod, {})
        for label, key in [
            ("improvements",    "improvements"),
            ("regressions",     "regressions"),
            ("persistent",      "persistent_errors"),
        ]:
            df_sheet = ra.get(key, pd.DataFrame())
            if not df_sheet.empty:
                sheet_name = f"{mod[:8]}_{label[:8]}"
                df_sheet.to_excel(writer, sheet_name=sheet_name, index=False)

    # 'Other' notes summary
    other_frames = []
    for mod in MODULES:
        mc = GOLD_COLUMNS[mod]
        other_col = mc.get("other_note_col")
        if other_col and other_col in gold_df.columns:
            sub = gold_df[
                gold_df[other_col].notna()
                & (gold_df[other_col].astype(str).str.strip() != "")
                & (gold_df[other_col].astype(str).str.lower() != "nan")
            ][[UID_COL, mc["code_col"], other_col]].copy()
            sub.insert(0, "module", mod)
            other_frames.append(sub)
    if other_frames:
        pd.concat(other_frames, ignore_index=True).to_excel(
            writer, sheet_name="other_notes", index=False
        )

print(f"✓ Comparison report saved → {COMPARISON_REPORT}")
print(f"  Sheets: summary + per-module improvements / regressions / persistent + other_notes")

# ── Final console summary ─────────────────────────────────────────────────
print("\n" + "═"*62)
print("FINAL SUMMARY")
print("═"*62)
for mod in MODULES:
    if mod not in baseline_metrics or mod not in enhanced_metrics:
        continue
    bm = _summary_metrics(baseline_metrics[mod])
    em = _summary_metrics(enhanced_metrics[mod])
    ra = record_analysis.get(mod, {})
    print(f"\n{mod}  (n={bm['n']})")
    print(f"  Accuracy   : {bm['accuracy']:.3f} → {em['accuracy']:.3f}  ({em['accuracy']-bm['accuracy']:+.3f})")
    print(f"  Macro F1   : {bm['macro_f1']:.3f} → {em['macro_f1']:.3f}  ({em['macro_f1']-bm['macro_f1']:+.3f})")
    print(f"  Macro P    : {bm['macro_precision']:.3f} → {em['macro_precision']:.3f}  ({em['macro_precision']-bm['macro_precision']:+.3f})")
    print(f"  Macro R    : {bm['macro_recall']:.3f} → {em['macro_recall']:.3f}  ({em['macro_recall']-bm['macro_recall']:+.3f})")
    print(f"  Improvements : {len(ra.get('improvements', []))}")
    print(f"  Regressions  : {len(ra.get('regressions', []))}  {'⚠️' if len(ra.get('regressions', [])) > 0 else '✓'}")
    print(f"  Persistent   : {len(ra.get('persistent_errors', []))}")
print()
print("Done.")